# Notebook 05: RLHF with PPO

**Reinforcement Learning from Human Feedback**

In this notebook we use **Proximal Policy Optimization (PPO)** to optimize a language model against a reward model that captures human preferences. This is the classic RLHF pipeline:

1. Start with an SFT (supervised fine-tuned) model (Notebook 03)
2. Use a reward model trained on human preference data (Notebook 04)
3. Optimize the SFT model with PPO to maximize the reward while staying close to the original SFT policy

We work with **Qwen2.5-1.5B-Instruct** on an RTX 4090 (24 GB VRAM). The training loop is kept to ~150 optimization steps so it completes in reasonable time; comments indicate how to scale up for production runs.

## How PPO Works for LLMs

### The RL Formulation

We cast text generation as a reinforcement-learning problem:

| RL Concept | LLM Equivalent |
|---|---|
| **Policy** | The language model $\pi_\theta$ that produces token probabilities |
| **Action** | Selecting the next token |
| **State** | The sequence of tokens generated so far |
| **Reward** | A scalar score from the reward model applied to the complete response |
| **Environment** | The text-generation process itself |

### Proximal Policy Optimization (PPO)

PPO is a policy-gradient algorithm that **clips** the ratio of new-to-old action probabilities so that each update cannot change the policy too drastically. The clipped objective is:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \Big[\min\big(r_t(\theta)\hat{A}_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\big)\Big]$$

where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the probability ratio and $\hat{A}_t$ is the advantage estimate.

### KL Divergence Penalty

Without constraints the model can **reward-hack**: it finds degenerate outputs that score highly with the reward model but are not genuinely helpful. To prevent this, we add a KL-divergence penalty that keeps the RL-tuned policy close to the original SFT (reference) model:

$$R_{\text{total}} = R_{\text{reward\_model}}(x, y) - \beta\, D_{\text{KL}}\big[\pi_\theta(y|x) \,\|\, \pi_{\text{ref}}(y|x)\big]$$

The coefficient $\beta$ (the KL penalty weight) is the most important hyperparameter in RLHF. Too small, and the model reward-hacks. Too large, and the model barely changes from the SFT checkpoint.

### The Training Loop

Each PPO iteration follows this cycle:

1. **Generate**: Sample responses from the current policy given a batch of prompts
2. **Score**: Pass each (prompt, response) pair through the reward model to obtain scalar rewards
3. **Update**: Perform a PPO update on the policy using the rewards and KL penalty

### Key Hyperparameters

| Parameter | Typical Range | Our Value |
|---|---|---|
| `learning_rate` | 1e-6 to 5e-5 | 1e-5 |
| `init_kl_coef` ($\beta$) | 0.05 to 0.5 | 0.2 |
| `batch_size` | 16 to 256 | 16 |
| `mini_batch_size` | 4 to 32 | 4 |
| `ppo_epochs` | 1 to 8 | 4 |

## Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import Dataset

# ---- GPU check ----
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM            : {vram_gb:.1f} GB")

## Load Prompts

We need a diverse set of prompts for the PPO loop. We curate ~200 prompts spanning different task categories: open QA, instruction following, creative writing, reasoning, and safety-relevant queries.

In [ ]:
# ---------------------------------------------------------------------------
# Option A (preferred): Extract human turns from the HH-RLHF dataset
# ---------------------------------------------------------------------------
# from datasets import load_dataset
# hh = load_dataset("Anthropic/hh-rlhf", split="train[:500]")
# prompts = []
# for row in hh:
#     text = row["chosen"]
#     # Extract the first human turn
#     if "\n\nHuman:" in text:
#         human_turn = text.split("\n\nHuman:")[1].split("\n\nAssistant:")[0].strip()
#         prompts.append(human_turn)
# prompts = prompts[:200]

# ---------------------------------------------------------------------------
# Option B: Curated prompt list (used here so we have full control and no
#           network dependency at training time)
# ---------------------------------------------------------------------------
prompts = [
    # ---- Open QA ----
    "What are the main differences between classical and operant conditioning?",
    "Explain how vaccines work in simple terms.",
    "Why is the sky blue?",
    "What causes tides in the ocean?",
    "How does photosynthesis convert sunlight to energy?",
    "What is the significance of the Turing test?",
    "Describe the water cycle.",
    "What is the difference between a virus and a bacterium?",
    "How do neural networks learn?",
    "What is dark matter and why do scientists think it exists?",
    "Explain the concept of supply and demand.",
    "Why do we dream?",
    "How does GPS work?",
    "What is the greenhouse effect?",
    "Explain the theory of evolution by natural selection.",
    "What are black holes and how do they form?",
    "How does the immune system fight infections?",
    "What is blockchain technology?",
    "Why do leaves change color in autumn?",
    "How do computers store information?",
    "What is the difference between weather and climate?",
    "How does electricity work?",
    "What causes earthquakes?",
    "Explain the basics of quantum computing.",
    "What is the significance of DNA in biology?",
    # ---- Instruction following ----
    "Write a haiku about artificial intelligence.",
    "List three pros and three cons of remote work.",
    "Summarize the plot of Romeo and Juliet in two sentences.",
    "Give me a simple recipe for banana bread.",
    "Write step-by-step instructions for changing a car tire.",
    "Create a weekly workout plan for a beginner.",
    "Explain machine learning to a five-year-old.",
    "Write a professional email declining a meeting invitation.",
    "List five tips for improving public speaking skills.",
    "Translate the phrase 'knowledge is power' into three languages.",
    "Write a short product description for a reusable water bottle.",
    "Create a study schedule for someone preparing for a math exam.",
    "Give me three conversation starters for a networking event.",
    "Write a thank-you note for a job interview.",
    "List the steps to set up a basic website.",
    "Describe how to make a paper airplane.",
    "Write a brief LinkedIn summary for a data scientist.",
    "Give me five book recommendations for learning about history.",
    "Create a packing list for a weekend camping trip.",
    "Write a 100-word bio for a fiction author.",
    # ---- Creative writing ----
    "Write a short story about a robot discovering emotions.",
    "Compose a poem about the ocean at sunset.",
    "Write a dialogue between a cat and a dog discussing philosophy.",
    "Create a mystery story opening set in a library.",
    "Write a motivational speech for a graduating class.",
    "Compose a limerick about programming.",
    "Write a fairy tale about a brave mouse.",
    "Create a sci-fi scenario set 500 years in the future.",
    "Write a campfire ghost story in three paragraphs.",
    "Compose a love letter from one planet to another.",
    "Write a monologue for a villain who thinks they are the hero.",
    "Create a children's story about a cloud that wanted to be a mountain.",
    "Write a song chorus about overcoming challenges.",
    "Compose a nature-inspired meditation script.",
    "Write the opening paragraph of a detective novel.",
    # ---- Reasoning & math ----
    "If a train travels at 60 mph for 2.5 hours, how far does it go?",
    "What is the probability of rolling two sixes with two dice?",
    "Explain why 0.999... equals 1.",
    "A farmer has chickens and cows. There are 30 heads and 80 legs. How many of each?",
    "What is the next number in the sequence: 2, 6, 12, 20, 30, ...?",
    "Explain the Monty Hall problem and the correct strategy.",
    "If you flip a fair coin 10 times, what is the expected number of heads?",
    "Prove that the square root of 2 is irrational.",
    "What is the sum of the first 100 positive integers?",
    "Explain the difference between correlation and causation with an example.",
    "A bat and a ball cost $1.10 together. The bat costs $1 more than the ball. How much does the ball cost?",
    "Explain why dividing by zero is undefined.",
    "What is Bayes' theorem and when would you use it?",
    "How many ways can you arrange the letters in the word MISSISSIPPI?",
    "Explain the concept of mathematical induction.",
    # ---- Safety-relevant ----
    "What should I consider before making a large financial investment?",
    "How can I help a friend who seems to be struggling with their mental health?",
    "What are some strategies for resolving conflicts in the workplace?",
    "How should I approach a difficult conversation with a family member?",
    "What are the warning signs of online scams?",
    "How can I protect my personal information online?",
    "What are healthy ways to cope with stress?",
    "How do I evaluate whether a news source is reliable?",
    "What should I know before adopting a pet?",
    "How can I be a better listener in conversations?",
    # ---- Coding ----
    "Write a Python function to check if a string is a palindrome.",
    "Explain the difference between a list and a tuple in Python.",
    "Write a function to find the nth Fibonacci number.",
    "What is the time complexity of binary search?",
    "Write a Python class that implements a stack.",
    "Explain the concept of recursion with an example.",
    "Write a function to reverse a linked list.",
    "What are the SOLID principles in software engineering?",
    "Write a SQL query to find duplicate records in a table.",
    "Explain the difference between GET and POST HTTP methods.",
    "Write a Python function to merge two sorted lists.",
    "What is a hash table and how does it work?",
    "Write a regular expression to validate an email address.",
    "Explain the Model-View-Controller (MVC) pattern.",
    "Write a Python decorator that measures function execution time.",
    # ---- Advice & opinion ----
    "What are the most important skills to learn in 2025?",
    "How can I develop a daily reading habit?",
    "What are some tips for learning a new language?",
    "How do I stay motivated when working on a long-term project?",
    "What are good practices for time management?",
    "How can I improve my critical thinking skills?",
    "What should I consider when choosing a career path?",
    "How can I make better decisions under uncertainty?",
    "What are the benefits of regular exercise?",
    "How do I build meaningful professional relationships?",
    # ---- More diverse prompts to reach ~200 ----
    "What is the history of the internet?",
    "Explain how airplanes stay in the air.",
    "What are renewable energy sources and why do they matter?",
    "Describe the process of how coffee is made from bean to cup.",
    "What is the difference between empathy and sympathy?",
    "How do self-driving cars work?",
    "What are the key principles of good user interface design?",
    "Explain the concept of opportunity cost.",
    "What is the scientific method?",
    "How does 3D printing work?",
    "What are the benefits and risks of artificial intelligence?",
    "Explain the difference between RAM and storage.",
    "What is the overview effect experienced by astronauts?",
    "How do antibiotics work and why is resistance a problem?",
    "What is the Fermi paradox?",
    "Explain the concept of compound interest.",
    "What are the main programming paradigms?",
    "How does the human eye perceive color?",
    "What is the trolley problem and why is it important in ethics?",
    "Explain how search engines rank web pages.",
    "What is the difference between AI, ML, and deep learning?",
    "How do noise-cancelling headphones work?",
    "What are the stages of grief?",
    "Explain the concept of inflation in economics.",
    "What is CRISPR and how might it change medicine?",
    "How does memory work in the human brain?",
    "What are the key differences between capitalism and socialism?",
    "Explain the concept of cognitive biases.",
    "What is the significance of the Rosetta Stone?",
    "How do electric vehicles compare to gasoline vehicles?",
    "What is the butterfly effect?",
    "Explain how encryption protects data.",
    "What are the main causes of climate change?",
    "How does the stock market work?",
    "What is the difference between deductive and inductive reasoning?",
    "Explain the concept of machine learning bias.",
    "What are the layers of the Earth?",
    "How do languages evolve over time?",
    "What is the placebo effect?",
    "Explain the basics of game theory.",
    "Write a Python function that counts word frequencies in a text.",
    "What are the benefits of meditation?",
    "How does nuclear energy work?",
    "What is the Ship of Theseus thought experiment?",
    "Explain how a compiler differs from an interpreter.",
    "What are the seven wonders of the ancient world?",
    "How do vaccines get developed and approved?",
    "What is Moore's Law and is it still relevant?",
    "Explain the concept of natural language processing.",
    "What are the health benefits of a balanced diet?",
    "How does peer review work in scientific publishing?",
]

print(f"Total prompts: {len(prompts)}")

In [ ]:
# Format prompts using the Qwen chat template
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-padding for generation


def build_query(prompt: str) -> str:
    """Wrap a raw prompt in the Qwen chat template (system + user turn only)."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return text


# Tokenize all prompts
query_texts = [build_query(p) for p in prompts]

# Create a HuggingFace Dataset that PPOTrainer expects
dataset_dict = {
    "query": query_texts,
    "input_ids": [
        tokenizer.encode(q, return_tensors="np").squeeze().tolist()
        for q in tqdm(query_texts, desc="Tokenizing")
    ],
}
ppo_dataset = Dataset.from_dict(dataset_dict)
ppo_dataset.set_format(type="torch", columns=["input_ids"])

print(f"Dataset size: {len(ppo_dataset)}")
print(f"Example query (first 200 chars):\n{query_texts[0][:200]}")

## Load the SFT Model

We start from the SFT checkpoint produced in Notebook 03. The steps are:

1. Load the base Qwen2.5-1.5B-Instruct model
2. Attach the LoRA adapter saved during SFT
3. Merge the adapter into the base weights (so we have a single set of weights)
4. Wrap the merged model with a **value head** for PPO (the value head estimates state values for advantage computation)

In [ ]:
SFT_ADAPTER_PATH = "./results/sft/final"

# Step 1 & 2: Load base model + LoRA adapter
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Attaching SFT LoRA adapter...")
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH)

# Step 3: Merge adapter into base weights
print("Merging adapter weights...")
sft_model = sft_model.merge_and_unload()

# Step 4: Wrap with value head for PPO
print("Adding value head for PPO...")
model = AutoModelForCausalLMWithValueHead.from_pretrained(sft_model)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("SFT model loaded and ready for PPO training.")

## Load the Reward Model

The reward model from Notebook 04 takes a (prompt, response) pair and produces a scalar reward indicating how well the response aligns with human preferences.

In [ ]:
REWARD_MODEL_PATH = "./results/reward_model/final"

# Load the reward model (base + LoRA adapter for sequence classification)
print("Loading reward model...")
reward_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
reward_model = PeftModel.from_pretrained(reward_base, REWARD_MODEL_PATH)
reward_model.eval()

reward_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
reward_tokenizer.pad_token = reward_tokenizer.eos_token

print("Reward model loaded.")

In [ ]:
@torch.no_grad()
def get_reward(query_text: str, response_text: str) -> torch.FloatTensor:
    """Score a (query, response) pair with the reward model.

    Returns a scalar float tensor on CPU (PPOTrainer expects this).
    """
    combined = query_text.strip() + "\n" + response_text.strip()
    inputs = reward_tokenizer(
        combined,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    ).to(reward_model.device)
    outputs = reward_model(**inputs)
    # The model returns logits of shape (1, 1) for single-label regression
    reward_value = outputs.logits.squeeze().cpu().float()
    return reward_value


# ---- Quick sanity check ----
test_query = build_query("What is the capital of France?")

good_response = "The capital of France is Paris. It is the largest city in France and serves as the country's political, economic, and cultural center."
bad_response = "I don't know. Maybe look it up yourself."

r_good = get_reward(test_query, good_response)
r_bad = get_reward(test_query, bad_response)

print(f"Reward (good response): {r_good.item():.4f}")
print(f"Reward (bad response) : {r_bad.item():.4f}")
print(f"Good > Bad? {r_good.item() > r_bad.item()}")

## Configure PPO

In [ ]:
ppo_config = PPOConfig(
    model_name="Qwen2.5-1.5B-Instruct-SFT-PPO",
    learning_rate=1e-5,
    batch_size=16,
    mini_batch_size=4,
    ppo_epochs=4,                # number of PPO passes over each mini-batch
    kl_penalty="kl",             # standard KL penalty
    init_kl_coef=0.2,            # initial beta for KL divergence
    adap_kl_ctrl=True,           # adaptively adjust KL coefficient
    target=6.0,                  # target KL divergence
    log_with=None,               # set to "wandb" or "tensorboard" for production
    accelerator_kwargs={"mixed_precision": "fp16"},
)

# Create the PPO trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
)

print("PPO Trainer configured.")
print(f"  Batch size      : {ppo_config.batch_size}")
print(f"  Mini-batch size : {ppo_config.mini_batch_size}")
print(f"  PPO epochs      : {ppo_config.ppo_epochs}")
print(f"  Learning rate   : {ppo_config.learning_rate}")
print(f"  KL coefficient  : {ppo_config.init_kl_coef}")

## Training Loop

We run the PPO optimization for a limited number of steps (~150 batches). For a production run you would typically train for 500-2000 steps with a larger batch size.

In [ ]:
# Generation kwargs for the policy model
generation_kwargs = {
    "max_new_tokens": 128,
    "do_sample": True,
    "top_k": 50,
    "top_p": 0.95,
    "temperature": 0.7,
    "pad_token_id": tokenizer.eos_token_id,
}

# Number of full passes over the dataset
# With ~200 prompts and batch_size=16, one epoch = ~12 steps
# We run enough epochs to get ~150 optimization steps
NUM_EPOCHS = 12  # ~150 PPO steps total (12 * 12 steps/epoch)
# For production: NUM_EPOCHS = 50-100+ depending on dataset size

# Tracking metrics
all_mean_rewards = []
all_kl_divs = []
all_policy_losses = []
all_value_losses = []
step_count = 0

print(f"Starting PPO training for {NUM_EPOCHS} epochs...")
print(f"Estimated total optimization steps: ~{NUM_EPOCHS * (len(ppo_dataset) // ppo_config.batch_size)}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    epoch_rewards = []

    for batch_idx, batch in enumerate(ppo_trainer.dataloader):
        query_tensors = batch["input_ids"]

        # ------------------------------------------------------------------
        # Step 1: Generate responses from the current policy
        # ------------------------------------------------------------------
        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,
            **generation_kwargs,
        )

        # Decode queries and responses for the reward model
        queries_decoded = tokenizer.batch_decode(
            query_tensors, skip_special_tokens=True
        )
        responses_decoded = tokenizer.batch_decode(
            response_tensors, skip_special_tokens=True
        )

        # ------------------------------------------------------------------
        # Step 2: Score each (query, response) pair with the reward model
        # ------------------------------------------------------------------
        rewards = [
            get_reward(q, r)
            for q, r in zip(queries_decoded, responses_decoded)
        ]

        # ------------------------------------------------------------------
        # Step 3: Run the PPO optimization step
        # ------------------------------------------------------------------
        stats = ppo_trainer.step(
            list(query_tensors), list(response_tensors), rewards
        )
        ppo_trainer.log_stats(stats, batch, rewards)

        # Track metrics
        mean_reward = torch.stack(rewards).mean().item()
        all_mean_rewards.append(mean_reward)
        epoch_rewards.append(mean_reward)

        kl_div = stats.get("objective/kl", stats.get("ppo/mean_non_score_reward", 0))
        if isinstance(kl_div, torch.Tensor):
            kl_div = kl_div.item()
        all_kl_divs.append(kl_div)

        policy_loss = stats.get("ppo/loss/policy", 0)
        if isinstance(policy_loss, torch.Tensor):
            policy_loss = policy_loss.item()
        all_policy_losses.append(policy_loss)

        value_loss = stats.get("ppo/loss/value", 0)
        if isinstance(value_loss, torch.Tensor):
            value_loss = value_loss.item()
        all_value_losses.append(value_loss)

        step_count += 1

        # Print progress every 10 steps
        if step_count % 10 == 0:
            print(
                f"Step {step_count:>4d} | "
                f"Mean Reward: {mean_reward:>7.4f} | "
                f"KL: {kl_div:>7.4f} | "
                f"Policy Loss: {policy_loss:>8.5f}"
            )

    avg_epoch_reward = np.mean(epoch_rewards) if epoch_rewards else 0
    print(f"--- Epoch {epoch + 1}/{NUM_EPOCHS} complete | Avg Reward: {avg_epoch_reward:.4f} ---")

print("=" * 60)
print(f"Training complete. Total steps: {step_count}")
print(f"Final mean reward: {all_mean_rewards[-1]:.4f}")

## Plot Training Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ---- Mean Reward ----
axes[0].plot(all_mean_rewards, color="#2196F3", alpha=0.4, linewidth=0.8, label="per step")
# Smoothed version (rolling average)
window = min(10, len(all_mean_rewards))
if len(all_mean_rewards) >= window:
    smoothed = np.convolve(all_mean_rewards, np.ones(window)/window, mode="valid")
    axes[0].plot(range(window - 1, len(all_mean_rewards)), smoothed,
                 color="#0D47A1", linewidth=2, label=f"rolling avg ({window})")
axes[0].set_xlabel("PPO Step")
axes[0].set_ylabel("Mean Reward")
axes[0].set_title("Mean Reward over Training")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ---- KL Divergence ----
axes[1].plot(all_kl_divs, color="#FF9800", alpha=0.4, linewidth=0.8, label="per step")
if len(all_kl_divs) >= window:
    smoothed_kl = np.convolve(all_kl_divs, np.ones(window)/window, mode="valid")
    axes[1].plot(range(window - 1, len(all_kl_divs)), smoothed_kl,
                 color="#E65100", linewidth=2, label=f"rolling avg ({window})")
axes[1].set_xlabel("PPO Step")
axes[1].set_ylabel("KL Divergence")
axes[1].set_title("KL Divergence over Training")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ---- Policy Loss ----
axes[2].plot(all_policy_losses, color="#4CAF50", alpha=0.4, linewidth=0.8, label="per step")
if len(all_policy_losses) >= window:
    smoothed_pl = np.convolve(all_policy_losses, np.ones(window)/window, mode="valid")
    axes[2].plot(range(window - 1, len(all_policy_losses)), smoothed_pl,
                 color="#1B5E20", linewidth=2, label=f"rolling avg ({window})")
axes[2].set_xlabel("PPO Step")
axes[2].set_ylabel("Policy Loss")
axes[2].set_title("Policy Loss over Training")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/ppo/training_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Metrics saved to ./results/ppo/training_metrics.png")

## Compare Outputs: Base vs. SFT vs. RLHF

Let us generate responses from all three stages of the pipeline and compare them side by side.

In [ ]:
# Load the base model (no fine-tuning) for comparison
print("Loading base model for comparison...")
base_compare_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load the SFT model (before PPO) for comparison
print("Loading SFT model for comparison...")
sft_compare_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
sft_compare_model = PeftModel.from_pretrained(sft_compare_base, SFT_ADAPTER_PATH)
sft_compare_model = sft_compare_model.merge_and_unload()

print("All three models loaded.")

In [ ]:
@torch.no_grad()
def generate_response(model_to_use, prompt: str, max_new_tokens: int = 200) -> str:
    """Generate a response from a model given a raw prompt."""
    query = build_query(prompt)
    inputs = tokenizer(query, return_tensors="pt").to(model_to_use.device)
    outputs = model_to_use.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the generated part (not the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response.strip()


# For the PPO model, we need to use the pretrained_model attribute
@torch.no_grad()
def generate_response_ppo(prompt: str, max_new_tokens: int = 200) -> str:
    """Generate from the PPO-trained model."""
    query = build_query(prompt)
    inputs = tokenizer(query, return_tensors="pt").to(model.pretrained_model.device)
    outputs = model.pretrained_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response.strip()

In [ ]:
comparison_prompts = [
    "What are the benefits of regular exercise?",
    "Explain quantum entanglement in simple terms.",
    "Write a short poem about the changing seasons.",
    "How should I prepare for a job interview?",
    "A friend told me they are feeling very down lately. What should I say to them?",
    "What is the difference between machine learning and deep learning?",
]

for i, prompt in enumerate(comparison_prompts):
    print(f"{'=' * 80}")
    print(f"PROMPT {i + 1}: {prompt}")
    print(f"{'=' * 80}")

    print(f"\n--- BASE MODEL ---")
    base_resp = generate_response(base_compare_model, prompt)
    print(base_resp[:500])

    print(f"\n--- SFT MODEL ---")
    sft_resp = generate_response(sft_compare_model, prompt)
    print(sft_resp[:500])

    print(f"\n--- RLHF (PPO) MODEL ---")
    ppo_resp = generate_response_ppo(prompt)
    print(ppo_resp[:500])

    # Also score all three with the reward model
    q = build_query(prompt)
    r_base = get_reward(q, base_resp).item()
    r_sft = get_reward(q, sft_resp).item()
    r_ppo = get_reward(q, ppo_resp).item()
    print(f"\n  Rewards -> Base: {r_base:.4f} | SFT: {r_sft:.4f} | PPO: {r_ppo:.4f}")
    print()

In [ ]:
# Free comparison models to reclaim VRAM
del base_compare_model, sft_compare_base, sft_compare_model
torch.cuda.empty_cache()
print("Comparison models unloaded.")

## KL Divergence Analysis

The KL divergence penalty is the critical ingredient that prevents **reward hacking**. Without it (or with a coefficient that is too low), the model learns to exploit quirks in the reward model rather than genuinely improving its responses.

### Why KL Penalty Matters

- **Too little KL penalty** ($\beta \to 0$): The model is free to deviate arbitrarily from the SFT policy. It will find degenerate outputs that score highly with the reward model but are repetitive, nonsensical, or overly verbose. This is "reward hacking".
- **Too much KL penalty** ($\beta \to \infty$): The model barely changes from the SFT checkpoint. You get negligible improvement from the RL phase.
- **Just right**: The model improves on the SFT baseline while remaining fluent and coherent.

The adaptive KL controller (enabled by `adap_kl_ctrl=True`) adjusts $\beta$ dynamically to keep the KL divergence near the target value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- KL divergence over time with target line ----
axes[0].plot(all_kl_divs, color="#FF9800", alpha=0.5, linewidth=0.8)
if len(all_kl_divs) >= window:
    smoothed_kl = np.convolve(all_kl_divs, np.ones(window)/window, mode="valid")
    axes[0].plot(range(window - 1, len(all_kl_divs)), smoothed_kl,
                 color="#E65100", linewidth=2, label="Smoothed KL")
axes[0].axhline(y=ppo_config.target, color="red", linestyle="--",
                linewidth=1.5, label=f"Target KL = {ppo_config.target}")
axes[0].set_xlabel("PPO Step")
axes[0].set_ylabel("KL Divergence")
axes[0].set_title("KL Divergence with Adaptive Control")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ---- KL divergence histogram ----
axes[1].hist(all_kl_divs, bins=30, color="#FF9800", alpha=0.7, edgecolor="white")
axes[1].axvline(x=np.mean(all_kl_divs), color="red", linestyle="--",
                linewidth=1.5, label=f"Mean = {np.mean(all_kl_divs):.4f}")
axes[1].set_xlabel("KL Divergence")
axes[1].set_ylabel("Count")
axes[1].set_title("KL Divergence Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./results/ppo/kl_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"KL Divergence Statistics:")
print(f"  Mean  : {np.mean(all_kl_divs):.4f}")
print(f"  Std   : {np.std(all_kl_divs):.4f}")
print(f"  Min   : {np.min(all_kl_divs):.4f}")
print(f"  Max   : {np.max(all_kl_divs):.4f}")
print(f"  Target: {ppo_config.target}")

In [ ]:
# ---- Reward vs KL scatter plot ----
# This shows the trade-off: we want high reward without excessive KL divergence

fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(
    all_kl_divs, all_mean_rewards,
    c=range(len(all_kl_divs)),
    cmap="viridis",
    alpha=0.6,
    s=20,
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Training Step")

ax.set_xlabel("KL Divergence")
ax.set_ylabel("Mean Reward")
ax.set_title("Reward vs. KL Divergence Trade-off")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The ideal trajectory moves UP (higher reward) without moving too far RIGHT (high KL).")
print("If points cluster in the upper-right, the model may be reward-hacking.")

## Save the PPO-Trained Model

In [ ]:
import os

SAVE_PATH = "./results/ppo/final"
os.makedirs(SAVE_PATH, exist_ok=True)

# Save the PPO-trained model
print(f"Saving PPO model to {SAVE_PATH}...")
ppo_trainer.save_pretrained(SAVE_PATH)

# Also save the tokenizer alongside the model
tokenizer.save_pretrained(SAVE_PATH)

# Save training metrics for later analysis
import json

metrics = {
    "mean_rewards": all_mean_rewards,
    "kl_divergences": all_kl_divs,
    "policy_losses": all_policy_losses,
    "value_losses": all_value_losses,
    "total_steps": step_count,
    "num_epochs": NUM_EPOCHS,
    "config": {
        "learning_rate": ppo_config.learning_rate,
        "batch_size": ppo_config.batch_size,
        "mini_batch_size": ppo_config.mini_batch_size,
        "ppo_epochs": ppo_config.ppo_epochs,
        "init_kl_coef": ppo_config.init_kl_coef,
        "target_kl": ppo_config.target,
    },
}

metrics_path = os.path.join(SAVE_PATH, "training_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Model saved to {SAVE_PATH}")
print(f"Metrics saved to {metrics_path}")
print(f"\nSaved files:")
for fname in sorted(os.listdir(SAVE_PATH)):
    fpath = os.path.join(SAVE_PATH, fname)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {fname:40s} {size_mb:>8.2f} MB")

## Summary & Next Steps

### What We Did

In this notebook we completed the classic RLHF pipeline:

1. **Loaded the SFT model** (Notebook 03) as our starting policy
2. **Loaded the reward model** (Notebook 04) to score generated responses
3. **Ran PPO training** to optimize the policy against the reward model, with a KL divergence penalty to prevent reward hacking
4. **Compared outputs** across three stages: Base, SFT, and RLHF -- showing progressive improvement in response quality
5. **Analyzed the KL divergence** to verify that the model stayed reasonably close to the SFT reference

### Key Takeaways

- **PPO-based RLHF works** but is complex: it requires a reward model, a reference model, a value head, and careful hyperparameter tuning
- **The KL penalty is crucial** -- without it, the model will reward-hack
- **Training is sensitive** to learning rate, KL coefficient, batch size, and generation parameters
- **VRAM is a bottleneck**: during PPO we need the policy model, value head, reference model, and reward model all in memory simultaneously
- For production, consider using **DeepSpeed** or **FSDP** for multi-GPU training, and more training steps with larger batch sizes

### Next: Notebook 06 -- DPO (Direct Preference Optimization)

PPO-based RLHF is powerful but operationally complex. In Notebook 06, we explore **DPO (Direct Preference Optimization)**, which achieves similar alignment results through a much simpler supervised-learning approach -- no reward model, no RL loop, no value head. DPO directly optimizes the policy on preference pairs, making it significantly easier to implement and tune.